# Paper 4 — Resumable A-OKVQA execution
Runs the matched B0–B5 experiment, source/top-k/verifier/reranker/fusion ablations, controlled evidence perturbations, target-risk sensitivity, statistics and figures. **GPU execution has not been validated by the authoring environment.** Outputs are generated only by actual execution.

Select a CUDA GPU in Runtime → Change runtime type. Run the cells in order. Drive authorization is interactive. The extended study contains many model runs and may need multiple Colab sessions. Reopen this notebook and rerun to resume with the same settings.

This does not complete the entire manuscript: official OK-VQA evaluation, external matched baselines, additional corruption-seed repeats and independent evidence-support assessment remain outstanding. A 5% calibration target is not a guarantee of 5% held-out error.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json
CAL_N = 1000
EVAL_MAX = None  # 100 for a separately named development run; None for the full validation split
SEED = 2026
SCOPE = 'extended'  # 'main' limits execution to B0–B5 and target-risk sensitivity
MAX_PIXELS = 1003520  # Fixed before evaluation; changing it requires a new run directory
RUN_NAME = 'aokvqa_extended_v2'  # Use a NEW name if changing ANY configuration


## 1. Connect the GPU and persistent storage

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a CUDA GPU runtime before proceeding.'
print('GPU:', torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')
RUN = Path('/content/drive/MyDrive/Paper4Runs') / RUN_NAME
RUN.mkdir(parents=True, exist_ok=True)
print('Persistent run directory:', RUN)


## 2. Fetch and freeze the implementation

In [ ]:
REPO = 'https://github.com/junnubabu-ctrl/paper4-selective-kbvqa.git'
BRANCH = 'fix/calibration-jsonl-20260911'
ROOT = Path('/content/paper4-study-v2')
if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO, str(ROOT)], check=True)
commit_file = RUN / 'code_commit.txt'
if commit_file.exists():
    commit = commit_file.read_text().strip()
    subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', commit], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', commit], check=True)
else:
    commit = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip()
    commit_file.write_text(commit)
os.chdir(ROOT)
print('Frozen code commit:', commit)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[vlm,retrieval,dev,analysis]'], check=True)


## 3. Verify software and inspect the experiment plan

In [ ]:
subprocess.run([sys.executable, '-m', 'compileall', '-q', 'scripts', 'src'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)
subprocess.run([sys.executable, 'scripts/run_full_study.py', '--scope', SCOPE, '--plan'], check=True)


## 4. Run or resume the study
A failed stage stops execution and records its log. Resolve the error and rerun with the same configuration. Do not edit benchmark values or substitute development outputs.

In [ ]:
command = [sys.executable, 'scripts/run_full_study.py', '--out', str(RUN),
           '--scope', SCOPE, '--cal-n', str(CAL_N), '--seed', str(SEED),
           '--max-pixels', str(MAX_PIXELS)]
if EVAL_MAX is not None:
    command += ['--eval-max', str(EVAL_MAX)]
subprocess.run(command, check=True)


## 5. Inspect completion and download the evidence bundle
This cell can also be run after an interruption to export partial results. The archive excludes model weights, COCO images and retrieval caches. Check `completion.json` and `progress.json` before describing a run as complete.

In [ ]:
import zipfile
from google.colab import files
status = RUN / 'completion.json'
print(status.read_text() if status.exists() else 'PARTIAL: study has not completed.')
archive = Path('/content') / (RUN_NAME + '_results.zip')
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for path in RUN.rglob('*'):
        if path.is_file() and path.relative_to(RUN).parts[0] not in {'datasets', 'cache'}:
            z.write(path, arcname=str(path.relative_to(RUN)))
files.download(str(archive))
